# Travel Agent — SerpApi connection test

בדיקה ראשונית של חיפוש טיסות ומלונות. זו עדיין אינה מערכת לתכנון חופשה.

1. הירשמו ב־[SerpApi](https://serpapi.com/users/sign_up) והעתיקו את המפתח מה־[Dashboard](https://serpapi.com/manage-api-key).
2. ב־Colab בחרו **Runtime → Run all** והדביקו את המפתח רק בשדה הסיסמה המוסתר.
3. יופיעו עד חמש טיסות ועד חמישה מלונות, וכן סטטוס הבדיקה לכל שירות.

המפתח אינו כלול במחברת ואינו נכתב לקובץ התוצאות. ברירת המחדל היא שתי קריאות חיפוש בלבד, ללא ניסיונות חוזרים או הזמנות. הרצה נוספת מבצעת שתי קריאות נוספות.

**תרחיש:** שני מבוגרים, טיסה בכיוון אחד TLV → FCO, ארבעה לילות ברומא, מחירים באירו. התאריכים נקבעים אוטומטית לעוד 30 יום ואפשר לשנותם בתא הבא.

**גבולות הבדיקה:** אין טיסת חזור, אין בדיקת התאמה בין מועד הנחיתה לצ׳ק־אין ואין חישוב עלות חופשה. מחיר הטיסה מוצג כפי שהוחזר מהמקור, ללא הכפלה במספר נוסעים. מחירי המלונות מוצגים בנפרד ללילה ולשהייה; מידע חסר נשאר חסר.


In [ ]:
from datetime import date, timedelta, datetime, timezone
from getpass import getpass
from pathlib import Path
import json
import os
import requests
import pandas as pd
from IPython.display import display

# Change these values to test a different trip.
DEPARTURE_AIRPORT = "TLV"
ARRIVAL_AIRPORT = "FCO"
HOTEL_QUERY = "Hotels in Rome, Italy"
OUTBOUND_DATE = (date.today() + timedelta(days=30)).isoformat()
CHECK_IN_DATE = OUTBOUND_DATE
CHECK_OUT_DATE = (date.fromisoformat(CHECK_IN_DATE) + timedelta(days=4)).isoformat()
ADULTS = 2
CURRENCY = "EUR"

# Prefer a runtime environment variable; otherwise use a hidden prompt.
api_key = os.environ.get("SERPAPI_API_KEY", "").strip()
if not api_key:
    api_key = getpass("SerpApi API key (hidden): ").strip()
if not api_key:
    raise ValueError("Enter a SerpApi API key in the hidden field.")

print(f"One-way flight: {DEPARTURE_AIRPORT} → {ARRIVAL_AIRPORT}, {OUTBOUND_DATE}")
print(f"Hotels: {CHECK_IN_DATE} to {CHECK_OUT_DATE}; {ADULTS} adults; {CURRENCY}")


In [ ]:
def fetch_search(params, key):
    """Perform one request without exposing authenticated URLs or credentials."""
    try:
        response = requests.get(
            "https://serpapi.com/search.json",
            params={**params, "api_key": key},
            timeout=(10, 90),
        )
    except requests.RequestException:
        return None, "Connection failed or timed out. No automatic retry was made."
    if response.status_code != 200:
        return None, f"HTTP {response.status_code}. Check your key, quota and parameters."
    try:
        data = response.json()
    except ValueError:
        return None, "The response was not valid JSON."
    if not isinstance(data, dict):
        return None, "Unexpected response structure."
    if data.get("error"):
        return None, "SerpApi reported an error. Check the request in your dashboard."
    status = (data.get("search_metadata") or {}).get("status")
    if status not in (None, "Success"):
        return None, "Search did not finish successfully. Check the dashboard."
    return data, None


def flight_rows(data):
    """Keep returned prices unchanged and preserve all outbound segments."""
    rows = []
    for offer in (data.get("best_flights") or []) + (data.get("other_flights") or []):
        segments = offer.get("flights") or []
        if not segments:
            continue
        rows.append({
            "airlines": ", ".join(dict.fromkeys(
                s.get("airline") for s in segments if s.get("airline")
            )),
            "flight_numbers": ", ".join(s.get("flight_number", "") for s in segments),
            "departure": (segments[0].get("departure_airport") or {}).get("time"),
            "arrival": (segments[-1].get("arrival_airport") or {}).get("time"),
            "duration_minutes": offer.get("total_duration"),
            "stops": len(segments) - 1,
            "returned_price": offer.get("price"),
            "currency": CURRENCY,
            "trip_type": offer.get("type"),
        })
    return rows


def hotel_rows(data):
    """Keep unknown fields null rather than inventing prices or coordinates."""
    rows = []
    for hotel in data.get("properties") or []:
        rate = hotel.get("rate_per_night") or {}
        total = hotel.get("total_rate") or {}
        coords = hotel.get("gps_coordinates") or {}
        rows.append({
            "name": hotel.get("name"),
            "property_type": hotel.get("type"),
            "price_per_night": rate.get("extracted_lowest"),
            "price_for_stay": total.get("extracted_lowest"),
            "currency": CURRENCY,
            "rating": hotel.get("overall_rating"),
            "latitude": coords.get("latitude"),
            "longitude": coords.get("longitude"),
        })
    return rows


def summarize_status(rows, error, required_fields):
    if error:
        return "FAILED"
    if not rows:
        return "NO_RESULTS"
    if any(all(row.get(field) is not None for field in required_fields) for row in rows):
        return "OK"
    return "PARTIAL_DATA"


In [ ]:
flight_params = {
    "engine": "google_flights", "departure_id": DEPARTURE_AIRPORT,
    "arrival_id": ARRIVAL_AIRPORT, "outbound_date": OUTBOUND_DATE,
    "type": 2, "adults": ADULTS, "currency": CURRENCY, "hl": "en",
}
hotel_params = {
    "engine": "google_hotels", "q": HOTEL_QUERY,
    "check_in_date": CHECK_IN_DATE, "check_out_date": CHECK_OUT_DATE,
    "adults": ADULTS, "currency": CURRENCY, "hl": "en",
}

# Two requests, with no pagination, retries or booking calls.
snapshot = {"provider": "SerpApi", "scope": "connection_test", "results": {}}
for label, params, normalizer, required in [
    ("flights", flight_params, flight_rows, ["returned_price", "departure", "arrival"]),
    ("hotels", hotel_params, hotel_rows, ["name", "price_for_stay", "latitude", "longitude"]),
]:
    data, error = fetch_search(params, api_key)
    rows = normalizer(data) if data is not None else []
    metadata = (data or {}).get("search_metadata") or {}
    status = summarize_status(rows, error, required)
    print(f"\n{label.upper()}: {status}; {len(rows)} options returned")
    if error:
        print(error)
    elif not rows:
        print("No options returned. This is not proof that the connection is broken.")
    else:
        display(pd.DataFrame(rows).head(5))
    # Save only selected fields, never raw response URLs or the API key.
    snapshot["results"][label] = {
        "status": status, "error": error, "query": params,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "search_id": metadata.get("id"),
        "source_created_at": metadata.get("created_at"),
        "records": rows,
    }

output = Path("serpapi_connection_test.json")
output.write_text(json.dumps(snapshot, ensure_ascii=False, indent=2), encoding="utf-8")
del api_key
print(f"\nSaved normalized results to {output}. This file contains no API key.")
print("Share the two status lines and the displayed tables to review the result.")


## How to interpret the result

- **OK:** התקבלו אפשרויות עם שדות שימושיים לבדיקה הראשונית. עדיין אין אישור שהספק מתאים לכל דרישות הפרויקט.
- **PARTIAL_DATA:** יש תוצאות, אבל שדות חיוניים חסרים; אין להשלים אותם באמצעות ניחוש.
- **NO_RESULTS:** החיפוש הושלם ללא אפשרויות; כדאי לבדוק יעד ותאריכים לפני הסקת מסקנה על הספק.
- **FAILED:** יש בעיית חיבור, מפתח, מכסה או בקשה. בדקו את ה־Dashboard; המחברת לא מנסה שוב אוטומטית.

הבדיקה הבאה לאחר הצלחה: טיסות הלוך־חזור, התאמה להרכב משפחתי, משמעות המחיר עבור מספר הנוסעים, מסים ועמלות במלונות ושמירת נתונים להערכה עקבית.

### Provider documentation
- [Google Flights API](https://serpapi.com/google-flights-api)
- [Google Hotels API](https://serpapi.com/google-hotels-api)
- [Account and pricing](https://serpapi.com/pricing)

תוצאות החיפוש אינן הבטחה למחיר או לזמינות בעת ההזמנה.
